# Module 14: Exponential Smoothing in Plain Language

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Exponential smoothing is the first method in this series that is a **model**
rather than a rule of thumb, and it is still simple enough to explain in a
meeting.

The idea: keep a running estimate of where the series is, how fast it is
moving, and what each month of the year does to it. Update all three every
month, weighting recent observations more than old ones.

**About 20 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")


def split(s, end="2024-12", horizon=12):
    """Train on everything up to `end`, test on the next `horizon` months."""
    train = s.loc[:end]
    test = s.loc[pd.Timestamp(end) + pd.offsets.MonthBegin(1):][:horizon]
    return train, test


def mae(actual, pred):
    return float(np.mean(np.abs(np.asarray(actual, float) - np.asarray(pred, float))))

grandview = series("A012")
train, test = split(grandview, end="2024-12")

## 2. Three things to track, three parameters

| What it tracks | Parameter | What the parameter decides |
|---|---|---|
| **Level**, where the series is now | alpha | how fast the model accepts that the level has moved |
| **Trend**, how fast it is moving | beta | how fast it accepts a change in direction |
| **Season**, what each month does | gamma | how fast it accepts a change in the seasonal shape |

Each runs from 0 to 1. Near 0 the model is stubborn and ignores new data; near
1 it is jumpy and chases every month. The software estimates them by fitting
the training data, and what it chose is worth reading.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

hw = ExponentialSmoothing(train, trend="add", seasonal="add",
                          seasonal_periods=12,
                          initialization_method="estimated").fit()

for name, key in [("alpha, the level", "smoothing_level"),
                  ("beta, the trend", "smoothing_trend"),
                  ("gamma, the season", "smoothing_seasonal")]:
    print(f"  {name:22s} {hw.params[key]:.3f}")

All three come out at **zero**, and that is a finding rather than a failure.

A smoothing parameter of zero means the model looked at the data and concluded
that recent months carry **no extra information** about the level, the trend or
the seasonal shape. Everything is estimated once, from the whole training
period, and never revised. In effect it has fitted a straight trend with a
fixed set of monthly factors.

For Ashfell that is believable: the level drifts smoothly and the seasonal
pattern is stable, so a month that came in high is noise rather than news.
It also explains most of the improvement in the next section. Seasonal naive
estimates the seasonal pattern from **one** year. This model estimates it from
**six**, which is why it is more accurate even though it is doing nothing
adaptive at all.

A series where the level genuinely shifts would produce a large alpha instead,
and the parameters are the first place to look when a forecast behaves
unexpectedly.

## 3. Does it beat the baseline?

The only question that matters, from [Module 13](Module_13_Baseline_Forecasts.ipynb).

In [ ]:
pred = hw.forecast(12)
snaive = train.iloc[-12:].values

print(f"Holt Winters        average error {mae(test.values, pred.values):5.2f}")
print(f"same month last year average error {mae(test.values, snaive):5.2f}")
print(f"improvement: {100 * (1 - mae(test.values, pred.values) / mae(test.values, snaive)):.0f} percent")

A 40 percent improvement over the free answer. That is worth having, and it is
the sort of margin that justifies maintaining a model.

In [ ]:
out = pd.DataFrame({
    "what happened": test.values.astype(int),
    "Holt Winters": pred.values.round(1),
    "same month last year": snaive.astype(int),
}, index=[f"{d:%b}" for d in test.index])
out["error, Holt Winters"] = (out["what happened"] - out["Holt Winters"]).round(1)
out

## 4. Additive or multiplicative season

Additive says July adds a fixed number of incidents. Multiplicative says July
multiplies by a fixed factor. For a series whose level is changing, the second
is usually closer to the truth, and it matters more the further the level has
moved.

In [ ]:
variants = {
    "additive trend, additive season":
        ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12,
                             initialization_method="estimated").fit(),
    "additive trend, multiplicative season":
        ExponentialSmoothing(train, trend="add", seasonal="mul", seasonal_periods=12,
                             initialization_method="estimated").fit(),
    "no trend, additive season":
        ExponentialSmoothing(train, trend=None, seasonal="add", seasonal_periods=12,
                             initialization_method="estimated").fit(),
}
scores = pd.Series({k: mae(test.values, v.forecast(12).values) for k, v in variants.items()})
scores.sort_values().round(2).to_frame("average error")

Here they are within a fifth of an incident of each other, so the choice does
not matter much for this agency. Do not read that as a general result: it holds
because Ashfell's level has fallen only about a quarter over the training
period. Fit both and compare, every time.

A third option is to take logs first, which makes an additive model
multiplicative and keeps the forecast from going negative. That last property
matters for small agencies.

## 5. Where the intervals come from

The interval below is built from the spread of the model's own one step errors
on the training data. It is a **lower bound** on the real uncertainty, because
it assumes the estimated parameters are correct and the future behaves like the
past.

In [ ]:
sd = float(np.std(hw.resid, ddof=1))
lower, upper = pred - 1.96 * sd, pred + 1.96 * sd

inside = ((test.values >= lower.values) & (test.values <= upper.values))
print(f"residual spread: {sd:.1f} incidents")
print(f"nominal interval: plus or minus {1.96 * sd:.1f}")
print(f"months the interval actually covered: {inside.sum()} of 12 "
      f"({100 * inside.mean():.0f} percent against a nominal 95)")

Close to nominal here, which is reassuring and not guaranteed. Checking
coverage rather than assuming it is [Module 15](Module_15_Measuring_Forecast_Error.ipynb).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9.5, 4))
show = grandview.loc["2022":"2025-12"]
ax.plot(show.index, show.values, color="#8a8880", lw=1.4, label="what happened")
ax.fill_between(test.index, lower, upper, color="#2a78d6", alpha=0.15,
                label="95 percent interval")
ax.plot(test.index, pred.values, color="#2a78d6", lw=2.4, label="Holt Winters")
ax.plot(test.index, snaive, color="#eb6834", lw=2, ls="--",
        label="same month last year")
ax.axvline(pd.Timestamp("2025-01-01"), color="k", ls="--", lw=1)
ax.set_ylim(0, None)
ax.set_ylabel("use of force incidents")
ax.legend(fontsize=8, frameon=False, loc="lower left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

## 6. What it cannot do

| Limitation | Consequence |
|---|---|
| It has no explanatory variables | it cannot tell you why, only what comes next |
| It assumes the future resembles the past | a policy change mid forecast makes it wrong |
| It needs several full cycles | two years of data will not estimate a seasonal shape |
| It goes negative on small counts | use logs, or a count model, for small agencies |
| The intervals assume the parameters are right | real coverage is usually worse than nominal |

## Exercise

Fit Holt Winters to Tarnbridge with and without `robust` handling of the
June 2021 unrest month, by fitting once on the full series and once with that
month removed, and compare the forecasts.

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A002"

if AGENCY:
    s = series(AGENCY)
    clean = s[~((s.index.year == 2021) & (s.index.month == 6))]
    for label, source in [("with the unrest month", s), ("without it", clean)]:
        tr, te = split(source, end="2024-12")
        f = ExponentialSmoothing(tr, trend="add", seasonal="add", seasonal_periods=12,
                                 initialization_method="estimated").fit()
        print(f"{label:24s} average error {mae(te.values, f.forecast(12).values):5.2f}   "
              f"June factor {f.params['initial_seasons'][5]:+.2f}")
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A002"
```

Removing the month improves the 2025 forecast only modestly, from an average
error of **6.73 to 6.18**, because the event is three and a half years before
the forecast period.

The dramatic change is somewhere the error score cannot see it. The estimated
**June seasonal component goes from +28.0 to +8.4**. With the unrest month left
in, the model has absorbed most of a one week event into its belief about what
June does *every year*, and it will carry that belief into every June forecast
it ever makes.

A forecast can therefore be reasonably accurate while resting on a seasonal
pattern that is largely an artifact. The score alone would not have told you.
The general rule from Beginner
[Topic 11](../../Beginner/Topic_11_Outliers_And_Spikes.md) applies: identify the
documented events first, decide explicitly what to do with them, record the
decision, and check the fitted components rather than only the error.

</details>

---

**Next:** [Module 15, How Wrong Is the Forecast](Module_15_Measuring_Forecast_Error.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*